# RQ2 — VOV4 Bahnar Discovery

Notebook này chỉ lập danh sách bài tiếng Bahnar công khai trên VOV4 để chuẩn bị `U_real`.

- Nguồn: trang Bahnar công khai `https://vov4.vov.vn/bahnar`.
- Lấy mọi bài Bahnar `.vov4` trên landing page hiện tại, rồi giữ 30 bài có `published_date` mới nhất.
- Lấy metadata và lưu HTML từng bài được giữ để audit.
- Chưa tải audio, chưa đoán media URL, chưa pseudo-label, chưa ASR/MT, chưa training.
- Không mở frozen test và không sửa artifact RQ1.

Bước sau, khi notebook này PASS, mới là Media Probe để xem bài nào có audio công khai.


## 1. Install / import

`pandas` và `requests` đã có trong `requirements.txt` của thesis. Cell này chỉ cài `beautifulsoup4` khi môi trường local chưa có, rồi import.

Chạy bằng venv `bahnar-s2tt` trên macOS. Không cần Colab hay RunPod.


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("bs4") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"])

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, urlunparse
import hashlib
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display


/Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Config


In [2]:
def find_project_root() -> Path:
    """Repo root whether the kernel cwd is the thesis root or notebooks/."""
    node = Path.cwd().resolve()
    for cand in [node, *node.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "notebooks").is_dir() and (cand / "src").is_dir():
            return cand
    raise RuntimeError(f"Cannot locate bahnar-s2tt-thesis root from {node}")


PROJECT_ROOT = find_project_root()
START_URL = "https://vov4.vov.vn/bahnar"
MAX_CANDIDATES = 30
SLEEP_SECONDS = 1.5
FETCH_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = (2, 4, 8)
OUT_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_discovery"

USER_AGENT = (
    "BahnarS2TT-Thesis-Research/1.0 "
    "(+non-commercial academic discovery; metadata only)"
)

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "vi,en;q=0.8",
})

print("PROJECT_ROOT:", PROJECT_ROOT)
print("START_URL:", START_URL)
print("OUT_DIR:", OUT_DIR)


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
START_URL: https://vov4.vov.vn/bahnar
OUT_DIR: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_discovery


## 3. robots.txt check

Fail-closed: không crawl khi `robots.txt` không tải được, không phải file robots, hoặc không cho user-agent này đọc `START_URL`. Không bypass anti-bot, login hay DRM.


In [3]:
parsed_start = urlparse(START_URL)
ROBOTS_URL = f"{parsed_start.scheme}://{parsed_start.netloc}/robots.txt"

robots_resp = session.get(ROBOTS_URL, timeout=20)
print("robots.txt status:", robots_resp.status_code)

if robots_resp.status_code != 200 or not robots_resp.text.strip():
    raise RuntimeError(f"Không xác minh được robots.txt: HTTP {robots_resp.status_code}")
if "user-agent" not in robots_resp.text.lower():
    raise RuntimeError("Không xác minh được robots.txt: response không phải file robots")

from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
rp.parse(robots_resp.text.splitlines())

if not rp.can_fetch(USER_AGENT, START_URL):
    raise PermissionError("robots.txt không cho crawl START_URL")

print("Can fetch START_URL: True")


robots.txt status: 200
Can fetch START_URL: True


## 4. Helper functions


In [4]:
def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")


def canonicalize(url: str) -> str:
    """Absolute URL on the start host, fragment removed."""
    joined = urljoin(START_URL, url)
    parts = urlparse(joined)
    path = parts.path or "/"
    return urlunparse((parts.scheme.lower(), parts.netloc.lower(), path, "", parts.query, ""))


LOW_PRIORITY_CATEGORY = "video-clip-bai-joh-hori-video-clip-ca-nhac"
BLOCKED_MEDIA_EXT = (".jpg", ".jpeg", ".png", ".gif", ".webp", ".mp3", ".mp4", ".pdf", ".xml", ".rss")


def is_bahnar_article(url: str) -> bool:
    """Article page: https://vov4.vov.vn/bahnar/<category>/<slug>.vov4.

    Section indexes are /bahnar or /bahnar/<section>. Media files are not articles.
    This does not guess a media URL.
    """
    parts = urlparse(url)
    if parts.scheme not in {"http", "https"}:
        return False
    if parts.netloc.lower() != "vov4.vov.vn":
        return False
    if not parts.path.startswith("/bahnar/"):
        return False
    segments = [seg for seg in parts.path.split("/") if seg]
    if len(segments) < 3 or segments[0].lower() != "bahnar":
        return False
    category, slug = segments[1], segments[-1].lower()
    if not category or not slug:
        return False
    if slug.endswith(BLOCKED_MEDIA_EXT):
        return False
    if not slug.endswith(".vov4"):
        return False
    return True


def candidate_priority(category: str) -> str:
    """Mark music/video clips for a later Media Probe exclude. Keep them in the manifest."""
    if LOW_PRIORITY_CATEGORY in (category or "").lower():
        return "low"
    return "normal"


def source_id(url: str) -> str:
    """Deterministic id: VOV4_ + SHA1 of the canonical page URL."""
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest().upper()
    return "VOV4_" + digest


def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def get_html(url: str, timeout: int = 30):
    """Fetch one HTML page. Returns raw bytes and the canonical final URL."""
    if not rp.can_fetch(USER_AGENT, url):
        raise PermissionError(f"robots.txt không cho crawl: {url}")
    response = session.get(url, timeout=timeout, allow_redirects=True)
    response.raise_for_status()
    final_url = canonicalize(response.url)
    final = urlparse(final_url)
    if final.netloc.lower() != "vov4.vov.vn":
        raise RuntimeError(f"Redirect rời vov4.vov.vn: {response.url}")
    if not rp.can_fetch(USER_AGENT, final_url):
        raise PermissionError(f"robots.txt không cho crawl final URL: {final_url}")
    ctype = (response.headers.get("Content-Type") or "").lower()
    if "text/html" not in ctype and "application/xhtml" not in ctype:
        raise RuntimeError(f"Không phải HTML: {ctype} — {url}")
    return response.content, final_url


class FetchFailed(RuntimeError):
    def __init__(self, page_url: str, error_type: str, error_message: str, attempts: int):
        self.page_url = page_url
        self.error_type = error_type
        self.error_message = error_message
        self.attempts = int(attempts)
        super().__init__(f"{error_type} after {attempts} attempt(s): {error_message}")


FETCH_STATS = {"retry_count_total": 0}


def _is_transient_fetch_error(exc: BaseException) -> bool:
    if isinstance(exc, (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError)):
        return True
    response = getattr(exc, "response", None)
    code = getattr(response, "status_code", 0) or 0
    return code == 429 or 500 <= code <= 599


def fetch_html(url: str, timeout: int = 30):
    """Up to FETCH_ATTEMPTS sequential tries. Retry only transient errors."""
    last_error = None
    for attempt in range(1, FETCH_ATTEMPTS + 1):
        try:
            content, final_url = get_html(url, timeout=timeout)
            return content, final_url, attempt
        except FetchFailed:
            raise
        except Exception as exc:
            if not _is_transient_fetch_error(exc):
                raise FetchFailed(url, type(exc).__name__, str(exc), attempt) from exc
            last_error = exc
            if attempt >= FETCH_ATTEMPTS:
                break
            delay = RETRY_BACKOFF_SECONDS[attempt - 1]
            FETCH_STATS["retry_count_total"] += 1
            print(f"RETRY {attempt}/{FETCH_ATTEMPTS - 1} in {delay}s: {type(exc).__name__} {url}")
            time.sleep(delay)
    raise FetchFailed(
        url,
        type(last_error).__name__,
        str(last_error),
        FETCH_ATTEMPTS,
    )


def parse_date_to_iso(text: str) -> str:
    for pattern in (
        re.compile(r"\b(\d{1,2})/(\d{1,2})/(\d{4})\b"),
        re.compile(r"\b(\d{1,2})-(\d{1,2})-(\d{4})\b"),
    ):
        match = pattern.search(text or "")
        if not match:
            continue
        day, month, year = map(int, match.groups())
        try:
            return datetime(year, month, day).date().isoformat()
        except ValueError:
            continue
    return ""


def extract_title(soup: BeautifulSoup) -> str:
    for selector in ('meta[property="og:title"]', 'meta[name="twitter:title"]'):
        node = soup.select_one(selector)
        if node and node.get("content"):
            return normalize_text(node.get("content"))
    heading = soup.find("h1")
    return normalize_text(heading.get_text(" ", strip=True)) if heading else ""


def extract_description(soup: BeautifulSoup) -> str:
    for selector in (
        'meta[property="og:description"]',
        'meta[name="description"]',
        'meta[name="twitter:description"]',
    ):
        node = soup.select_one(selector)
        if node and node.get("content"):
            return normalize_text(node.get("content"))
    return ""


def _date_from_raw(raw: str) -> str:
    match = re.search(r"\b(\d{4})-(\d{2})-(\d{2})\b", raw or "")
    if match:
        return "-".join(match.groups())
    return parse_date_to_iso(raw)


def extract_published_date(soup: BeautifulSoup):
    """Return (YYYY-MM-DD or '', source).

    source is metadata, time, body_fallback, or missing.
    """
    for selector in (
        'meta[property="article:published_time"]',
        'meta[name="pubdate"]',
        'meta[name="publish-date"]',
    ):
        node = soup.select_one(selector)
        if node is None:
            continue
        parsed = _date_from_raw((node.get("content") or "").strip())
        if parsed:
            return parsed, "metadata"
    time_node = soup.select_one("time[datetime]")
    if time_node is not None:
        raw = (time_node.get("datetime") or time_node.get("content") or "").strip()
        parsed = _date_from_raw(raw)
        if parsed:
            return parsed, "time"
    parsed = parse_date_to_iso(normalize_text(soup.get_text(" ", strip=True)))
    if parsed:
        return parsed, "body_fallback"
    return "", "missing"


def sort_candidates_by_date(frame: pd.DataFrame) -> pd.DataFrame:
    """Valid published_date descending. Rows with no date stay at the end."""
    ranked = frame.copy()
    dated = ranked["published_date"].astype(str).str.strip().ne("")
    ranked["_dated"] = dated
    ranked["_date_key"] = ranked["published_date"].where(dated, "")
    ranked = ranked.sort_values(
        by=["_dated", "_date_key"],
        ascending=[False, False],
        kind="mergesort",
    )
    return ranked.drop(columns=["_dated", "_date_key"]).reset_index(drop=True)


def category_slug(url: str) -> str:
    segments = [seg for seg in urlparse(url).path.split("/") if seg]
    if len(segments) >= 3 and segments[0].lower() == "bahnar":
        return segments[1]
    return ""


## 5. Discover candidate URLs

Chỉ đọc landing page Bahnar hiện tại. Giữ mọi URL bài `.vov4`, chưa cắt 30. Không sang trang khác và không request song song.


In [5]:
FETCH_STATS["retry_count_total"] = 0
landing_bytes, _landing_url, _landing_attempts = fetch_html(START_URL)
landing_soup = BeautifulSoup(landing_bytes, "html.parser")
time.sleep(SLEEP_SECONDS)

links = []
seen = set()
for anchor in landing_soup.find_all("a", href=True):
    url = canonicalize(anchor["href"])
    if not is_bahnar_article(url) or url in seen:
        continue
    seen.add(url)
    links.append(url)

print("Discovered:", len(links))
if not links:
    sample = []
    for anchor in landing_soup.find_all("a", href=True):
        url = canonicalize(anchor["href"])
        path = urlparse(url).path
        if urlparse(url).netloc.lower() == "vov4.vov.vn" and path not in sample:
            sample.append(path)
        if len(sample) >= 20:
            break
    print("Không có URL bài /bahnar/<category>/<slug>.vov4. Sample path trên trang:")
    for path in sample:
        print(" ", path)
    raise RuntimeError("VOV4 discovery found 0 Bahnar article URLs")

links[:5]


Discovered: 86


['https://vov4.vov.vn/bahnar/nor-pore-phat-thanh/todrong-kotong-ang-nar-2492026-529441.vov4',
 'https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/chanh-an-honih-tom-xek-tolang-teh-dak-nguyen-van-quang-tobop-poma-donuh-ham-bongai-tah-hla-lam-dong-529433.vov4',
 'https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/hop-akom-podrong-kio-trong-chih-tolech-hla-boar-tobat-potoch-bo-jang-dar-lang-lang-holen-mang-ma-3-ham-khul-kodra-chep-pogor-dang-honih-bo-jang-teh-dak-529404.vov4',
 'https://vov4.vov.vn/bahnar/choh-jang-sa-nha-nong/honhak-choh-potam-kojap-truh-ham-kon-polei-potam-sau-rieng-cheh-phe-tay-nguyen-529398.vov4',
 'https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/rim-toring-ko-tay-nguyen-tojra-ham-mi-kial-dak-hobong-529430.vov4']

## 6. Fetch metadata

Mỗi bài trên landing page là một request, rồi nghỉ `SLEEP_SECONDS`. Chưa cắt 30 ở bước này. HTML gốc được giữ trong bộ nhớ và chỉ ghi ra đĩa sau khi manifest qua gate.


In [6]:
rows = []
html_by_id = {}
seen_pages = set()
fetch_failures = []
n_discovered_urls = len(links)
n_fetched_ok = 0
n_fetch_failed = 0

for index, url in enumerate(links, 1):
    print(f"[{index}/{len(links)}] {url}")
    try:
        html_bytes, final_url, _attempts = fetch_html(url)
        if not is_bahnar_article(final_url):
            raise FetchFailed(url, "RuntimeError", f"Final URL is not a Bahnar article: {final_url}", 1)
        n_fetched_ok += 1
        if final_url in seen_pages:
            print("WARN: duplicate final URL", final_url)
        else:
            seen_pages.add(final_url)
            soup = BeautifulSoup(html_bytes, "html.parser")
            sid = source_id(final_url)
            published_date, published_date_source = extract_published_date(soup)
            cat = category_slug(final_url)
            html_by_id[sid] = html_bytes
            rows.append({
                "source_id": sid,
                "source": "VOV4",
                "page_url": final_url,
                "title": extract_title(soup),
                "published_date": published_date,
                "published_date_source": published_date_source,
                "category_slug": cat,
                "candidate_priority": candidate_priority(cat),
                "description": extract_description(soup),
                "page_sha256": hashlib.sha256(html_bytes).hexdigest(),
                "discovered_at_utc": utc_now(),
                "status": "candidate",
                "notes": "",
            })
    except FetchFailed as exc:
        n_fetch_failed += 1
        fetch_failures.append({
            "page_url": exc.page_url,
            "error_type": exc.error_type,
            "error_message": exc.error_message,
            "attempts": exc.attempts,
        })
        print("WARN:", exc.error_type, exc.error_message)
    except Exception as exc:
        n_fetch_failed += 1
        fetch_failures.append({
            "page_url": url,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "attempts": 1,
        })
        print("WARN:", type(exc).__name__, str(exc))
    time.sleep(SLEEP_SECONDS)

df = pd.DataFrame(rows)
print(
    "fetched_ok:", n_fetched_ok,
    "fetch_failed:", n_fetch_failed,
    "retries:", FETCH_STATS["retry_count_total"],
    "rows:", len(df),
)
df.head()


[1/86] https://vov4.vov.vn/bahnar/nor-pore-phat-thanh/todrong-kotong-ang-nar-2492026-529441.vov4
[2/86] https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/chanh-an-honih-tom-xek-tolang-teh-dak-nguyen-van-quang-tobop-poma-donuh-ham-bongai-tah-hla-lam-dong-529433.vov4
[3/86] https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/hop-akom-podrong-kio-trong-chih-tolech-hla-boar-tobat-potoch-bo-jang-dar-lang-lang-holen-mang-ma-3-ham-khul-kodra-chep-pogor-dang-honih-bo-jang-teh-dak-529404.vov4
[4/86] https://vov4.vov.vn/bahnar/choh-jang-sa-nha-nong/honhak-choh-potam-kojap-truh-ham-kon-polei-potam-sau-rieng-cheh-phe-tay-nguyen-529398.vov4
[5/86] https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/rim-toring-ko-tay-nguyen-tojra-ham-mi-kial-dak-hobong-529430.vov4
[6/86] https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/kon-polei-jang-mir-dak-lak-podap-gah-todrong-tech-rat-tomam-dram-lom-komai-koso-529431.vov4
[7/86] https://vov4.vov.vn/bahnar/kotong-a

,source_id,source,page_url,title,published_date,published_date_source,category_slug,candidate_priority,description,page_sha256,discovered_at_utc,status,notes
0,VOV4_23A5518B701085121391274F43D7F0A6854AF4A5,VOV4,https://vov4.vov.vn/bahnar/nor-pore-phat-thanh...,Tơdrong kơtơ̆ng ang năr 24.9.2026,2026-09-24,body_fallback,nor-pore-phat-thanh,normal,VOV.Bahnar - Tơdrong kơtơ̆ng ang năr ou đei dô...,05a1b1c9f05be99db710a72786a93c1ac283fccc694839...,2026-09-24T16:05:22Z,candidate,
1,VOV4_2D39DE6994374745A90B2F8CCC41A43ABF111EF0,VOV4,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...,Chánh án Hơnih tơm xek tơlang teh đak Nguyễn V...,2026-09-24,body_fallback,kotong-ang-lom-topol-thoi-su-xa-hoi,normal,"VOV.Bahnar - Pơgê năr 23/9, Bí thư Trung ương ...",752affa8ad9912077498c3f9c8c2f94c10b0d8c4db9f29...,2026-09-24T16:05:24Z,candidate,
2,VOV4_4F3554FC564400008578B69EB55E4D529F2A6E0E,VOV4,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...,Hop akŏm pơdrơ̆ng kiơ̆ trong chih tơlĕch Hla b...,2026-09-24,body_fallback,kotong-ang-lom-topol-thoi-su-xa-hoi,normal,"VOV.Bahnar - Pơgê năr 23/9, tơ̆ Hơnih bơ̆ jang...",4ebb0570f3a955aab26d39cf0196c98273d155c8572fd5...,2026-09-24T16:05:26Z,candidate,
3,VOV4_B6C2E19EE2E2E4F8454124A4F5E7D2687EFA78D3,VOV4,https://vov4.vov.vn/bahnar/choh-jang-sa-nha-no...,Hơnhăk choh pơtăm kơjăp truh hăm kon pơlei pơt...,2026-09-24,body_fallback,choh-jang-sa-nha-nong,normal,VOV.Bahnar - Hơnih pơtrŭt choh jang xa kơ teh ...,4165e143fb44fd8cd6e8c2e633b3f272847a5b492560fc...,2026-09-24T16:05:28Z,candidate,
4,VOV4_E13F09EA9BD4230FFC98E4F7F7DC8448DFD71104,VOV4,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...,Rim tơring kơ Tây Nguyên tơjră hăm 'mi kial đa...,2026-09-24,body_fallback,kotong-ang-lom-topol-thoi-su-xa-hoi,normal,VOV.Bahnar – 'Mi tih đunh năr pơm đak lơ̆p jrŭ...,0ba71f4aedfdf0818ffede134a2f4160b4bb04a2c56233...,2026-09-24T16:05:30Z,candidate,


## 7. Save manifest

Sort và giữ 30 bài chỉ khi mọi URL đã fetch thành công. Nếu còn failure, ghi `vov4_fetch_failures.jsonl`, summary là `PARTIAL_VOV4_DISCOVERY`, và không chọn top 30.


In [7]:
def assert_dates_sorted_desc(frame: pd.DataFrame) -> None:
    """Valid dates are descending. A missing date cannot appear before a valid date."""
    seen_missing = False
    valid_dates = []
    for raw in frame["published_date"].astype(str).tolist():
        value = raw.strip()
        if not value or value.lower() == "nan":
            seen_missing = True
            continue
        if seen_missing:
            raise RuntimeError("row without published_date is not at the end")
        valid_dates.append(value)
    if valid_dates != sorted(valid_dates, reverse=True):
        raise RuntimeError("published_date is not sorted descending")


def assert_discovery_frame(frame: pd.DataFrame) -> None:
    if len(frame) == 0:
        raise RuntimeError("VOV4 discovery found 0 Bahnar article URLs")
    if len(frame) > MAX_CANDIDATES:
        raise RuntimeError(f"manifest has {len(frame)} rows, cap is {MAX_CANDIDATES}")
    if not frame["source"].eq("VOV4").all():
        raise RuntimeError("source phải là VOV4")
    if frame["page_url"].duplicated().any():
        raise RuntimeError("page_url bị trùng")
    if frame["source_id"].duplicated().any():
        raise RuntimeError("source_id bị trùng")
    hosts = frame["page_url"].map(lambda url: urlparse(url).netloc.lower())
    if not hosts.eq("vov4.vov.vn").all():
        raise RuntimeError("page_url không thuộc vov4.vov.vn")
    paths = frame["page_url"].map(lambda url: urlparse(url).path.lower())
    if not paths.map(lambda path: path.startswith("/bahnar/") and path.endswith(".vov4")).all():
        raise RuntimeError("page_url không phải bài /bahnar/<category>/<slug>.vov4")
    expected_ids = frame["page_url"].map(source_id)
    if not expected_ids.equals(frame["source_id"]):
        raise RuntimeError("source_id không phải SHA1 của page_url")
    sources = set(frame["published_date_source"].astype(str))
    if not sources <= {"metadata", "time", "body_fallback", "missing"}:
        raise RuntimeError(f"published_date_source không hợp lệ: {sources}")
    priorities = set(frame["candidate_priority"].astype(str))
    if not priorities <= {"normal", "low"}:
        raise RuntimeError(f"candidate_priority không hợp lệ: {priorities}")
    assert_dates_sorted_desc(frame)


if len(df) == 0 and n_fetch_failed == 0:
    raise RuntimeError("VOV4 discovery fetched 0 Bahnar articles")

OUT_DIR.mkdir(parents=True, exist_ok=True)
failures_path = OUT_DIR / "vov4_fetch_failures.jsonl"
summary_path = OUT_DIR / "summary.json"
retry_count_total = int(FETCH_STATS["retry_count_total"])

if n_fetch_failed != 0:
    failure_lines = [json.dumps(row, ensure_ascii=False) for row in fetch_failures]
    failures_path.write_text("\n".join(failure_lines) + "\n", encoding="utf-8")
    summary = {
        "status": "PARTIAL_VOV4_DISCOVERY",
        "n_candidates": 0,
        "n_discovered_urls": int(n_discovered_urls),
        "n_fetched_ok": int(n_fetched_ok),
        "n_fetch_failed": int(n_fetch_failed),
        "n_candidates_final": 0,
        "n_with_date": 0,
        "n_low_priority": 0,
        "retry_count_total": retry_count_total,
        "generated_at_utc": utc_now(),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("FAILURES:", failures_path)
    raise RuntimeError(
        f"PARTIAL_VOV4_DISCOVERY: {n_fetch_failed} fetch failure(s); top 30 was not selected"
    )

if failures_path.exists():
    failures_path.unlink()

ranked = sort_candidates_by_date(df)
df = ranked.head(MAX_CANDIDATES).reset_index(drop=True)
kept_ids = set(df["source_id"].astype(str))
html_by_id = {sid: blob for sid, blob in html_by_id.items() if sid in kept_ids}
assert_discovery_frame(df)

raw_dir = OUT_DIR / "raw_html"
raw_dir.mkdir(parents=True, exist_ok=True)
for old_html in raw_dir.glob("*.html"):
    old_html.unlink()
for sid, html_bytes in html_by_id.items():
    (raw_dir / f"{sid}.html").write_bytes(html_bytes)

csv_path = OUT_DIR / "vov4_candidates.csv"
jsonl_path = OUT_DIR / "vov4_candidates.jsonl"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

dated = df["published_date"].astype(str).str.strip().ne("") & df["published_date"].astype(str).str.lower().ne("nan")
summary = {
    "status": "SUCCESS_VOV4_DISCOVERY",
    "n_candidates": int(len(df)),
    "n_discovered_urls": int(n_discovered_urls),
    "n_fetched_ok": int(n_fetched_ok),
    "n_fetch_failed": int(n_fetch_failed),
    "n_candidates_final": int(len(df)),
    "n_with_date": int(dated.sum()),
    "n_low_priority": int(df["candidate_priority"].eq("low").sum()),
    "retry_count_total": retry_count_total,
    "generated_at_utc": utc_now(),
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("CSV:", csv_path)
print("RAW_HTML:", raw_dir)


{
  "status": "SUCCESS_VOV4_DISCOVERY",
  "n_candidates": 30,
  "n_discovered_urls": 86,
  "n_fetched_ok": 86,
  "n_fetch_failed": 0,
  "n_candidates_final": 30,
  "n_with_date": 30,
  "n_low_priority": 3,
  "retry_count_total": 0,
  "generated_at_utc": "2026-09-24T16:08:15Z"
}
CSV: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_discovery/vov4_candidates.csv
RAW_HTML: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_discovery/raw_html


## 8. Quick audit

Chỉ xem manifest. Chưa mở audio.


In [8]:
audit_cols = [
    "source_id",
    "published_date",
    "published_date_source",
    "candidate_priority",
    "category_slug",
    "title",
    "page_url",
]
display(df[audit_cols].head(MAX_CANDIDATES))


,source_id,published_date,published_date_source,candidate_priority,category_slug,title,page_url
0,VOV4_23A5518B701085121391274F43D7F0A6854AF4A5,2026-09-24,body_fallback,normal,nor-pore-phat-thanh,Tơdrong kơtơ̆ng ang năr 24.9.2026,https://vov4.vov.vn/bahnar/nor-pore-phat-thanh...
1,VOV4_2D39DE6994374745A90B2F8CCC41A43ABF111EF0,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Chánh án Hơnih tơm xek tơlang teh đak Nguyễn V...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
2,VOV4_4F3554FC564400008578B69EB55E4D529F2A6E0E,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Hop akŏm pơdrơ̆ng kiơ̆ trong chih tơlĕch Hla b...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
3,VOV4_B6C2E19EE2E2E4F8454124A4F5E7D2687EFA78D3,2026-09-24,body_fallback,normal,choh-jang-sa-nha-nong,Hơnhăk choh pơtăm kơjăp truh hăm kon pơlei pơt...,https://vov4.vov.vn/bahnar/choh-jang-sa-nha-no...
4,VOV4_E13F09EA9BD4230FFC98E4F7F7DC8448DFD71104,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Rim tơring kơ Tây Nguyên tơjră hăm 'mi kial đa...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
5,VOV4_11AC992B7036F44651DC941869900437CA99DFF2,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Kon pơlei jang mir Đăk Lăk pơđăp gah tơdrong t...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
6,VOV4_4E064FE2CE9F07F56C8C6F0B757161248F3E4BCD,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Tổng Bí thư Kơdră teh đak bơ̆n Tô Lâm iung pơm...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
7,VOV4_3AACF99A3ADFB771059750FD550F7ED2BD427FB9,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,Chánh án Hơnih xek tơlang tơm Nguyễn Văn Quảng...,https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
8,VOV4_18412C2E196AE5AD50187C397998FBDAE4C9EDDD,2026-09-24,body_fallback,normal,kotong-ang-lom-topol-thoi-su-xa-hoi,"Tổng Bí thư, Kơdră chĕp pơgơ̆r teh đak Tô Lâm ...",https://vov4.vov.vn/bahnar/kotong-ang-lom-topo...
9,VOV4_C108541B901E319D5EAC87F59D9FE49489DF174A,2026-09-24,body_fallback,normal,trong-jang-teh-dak-tojur-an-pang-todrong-horih...,Vang pơjing tơdrong pơchơt pơhiơ̆ Trung thu ăn...,https://vov4.vov.vn/bahnar/trong-jang-teh-dak-...
